<a href="https://colab.research.google.com/github/Ryan56-56/porject2ML/blob/main/project2(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import imageio.v2 as imageio
from tqdm import tqdm


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:

BATCH_SIZE      = 128
LR_HEAD         = 0.01      # higher LR for classifier head
LR_BACKBONE     = 0.001     # lower LR for backbone
EPOCHS          = 80        # 50–80 is enough with these fixes
UNFREEZE_EPOCH  = 10
WEIGHT_DECAY    = 5e-4

train_path = "/content/drive/MyDrive/CIFAR-10-images-master/CIFAR-10-images-master/CIFAR-10-images-master/train"
test_path  = "/content/drive/MyDrive/CIFAR-10-images-master/CIFAR-10-images-master/CIFAR-10-images-master/test"

CKPT_DIR = "/content/CNN_model_CIFAR10"
os.makedirs(CKPT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [ ]:
train_transform = T.Compose([
    T.ToPILImage(),
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.2, 0.2, 0.2, 0.1),
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465],
                [0.2470, 0.2435, 0.2616]),
    T.RandomErasing(p=0.2)
])

test_transform = T.Compose([
    T.ToPILImage(),
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465],
                [0.2470, 0.2435, 0.2616])
])

class CIFARFolderDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.samples = []
        self.labels = []
        self.classes = sorted(os.listdir(root))

        print(f"Preloading images from {root} into RAM...")
        for label_idx, folder in enumerate(self.classes):
            folder_path = os.path.join(root, folder)
            for img_name in os.listdir(folder_path):
                img_path = os.path.join(folder_path, img_name)
                img = imageio.imread(img_path)
                self.samples.append(img)
                self.labels.append(label_idx)
        print(f"Loaded {len(self.samples)} images into RAM.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img = self.samples[idx]
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

train_ds = CIFARFolderDataset(train_path, transform=train_transform)
test_ds  = CIFARFolderDataset(test_path,  transform=test_transform)

train_dl = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    persistent_workers=False
)

test_dl = DataLoader(
    test_ds,
    batch_size=256,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    persistent_workers=False
)


Preloading images from /content/drive/MyDrive/CIFAR-10-images-master/CIFAR-10-images-master/CIFAR-10-images-master/train into RAM...


In [ ]:
class ResNet50_FT(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

        # Adapt for CIFAR-10 (32x32)
        self.model.conv1 = nn.Conv2d(
            3, 64, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.model.maxpool = nn.Identity()

        # Freeze backbone initially
        for param in self.model.parameters():
            param.requires_grad = False

        # New classifier head
        in_features = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def unfreeze_backbone(self):
        for name, param in self.model.named_parameters():
            if "fc" not in name:
                param.requires_grad = True

    def forward(self, x):
        return self.model(x)

model = ResNet50_FT().to(device)
model = model.to(memory_format=torch.channels_last)
model = torch.compile(model)


In [ ]:
# Separate params
head_params = list(model.model.fc.parameters())
backbone_params = [p for n, p in model.named_parameters() if "model.fc" not in n]

# Backbone stays frozen until UNFREEZE_EPOCH
for p in backbone_params:
    p.requires_grad = False

opt = optim.SGD(
    [
        {"params": head_params, "lr": LR_HEAD},
        {"params": backbone_params, "lr": LR_BACKBONE},
    ],
    momentum=0.9,
    weight_decay=WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.amp.GradScaler("cuda")

resume_path = os.path.join(CKPT_DIR, "last_epoch_resnet50.pth")
start_epoch = 0
backbone_unfrozen = False

if os.path.exists(resume_path):
    print("Resuming from last checkpoint...")
    checkpoint = torch.load(resume_path)

    loaded_state_dict = checkpoint["model"]
    new_state_dict = {}
    for k, v in loaded_state_dict.items():
        if k.startswith("_orig_mod."):
            new_key = k[len("_orig_mod."):]
        else:
            new_key = k
        new_state_dict[new_key] = v

    if hasattr(model, '_orig_mod'):
        model._orig_mod.load_state_dict(new_state_dict)
    else:
        model.load_state_dict(new_state_dict)

    opt.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])
    scaler.load_state_dict(checkpoint["scaler"])

    start_epoch = checkpoint["epoch"] + 1
    backbone_unfrozen = checkpoint.get("backbone_unfrozen", False)
    print(f"Resumed at epoch {start_epoch}, backbone_unfrozen={backbone_unfrozen}")


In [ ]:
def train_one_epoch(epoch):
    model.train()
    running_loss = 0.0
    loop = tqdm(train_dl, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)

    for xb, yb in loop:
        xb = xb.to(device, non_blocking=True, memory_format=torch.channels_last)
        yb = yb.to(device, non_blocking=True)

        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            preds = model(xb)
            loss = loss_fn(preds, yb)

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        running_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} | Loss: {running_loss/len(train_dl):.4f}")


def evaluate():
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for xb, yb in test_dl:
            xb = xb.to(device, non_blocking=True, memory_format=torch.channels_last)
            preds = model(xb).argmax(1).cpu()

            preds_all.extend(preds.numpy())
            labels_all.extend(yb.numpy())

    acc = accuracy_score(labels_all, preds_all)
    prec = precision_score(labels_all, preds_all, average='weighted', zero_division=0)
    rec = recall_score(labels_all, preds_all, average='weighted', zero_division=0)
    f1 = f1_score(labels_all, preds_all, average='weighted', zero_division=0)

    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 Score:  {f1:.4f}")

    return f1


In [ ]:
best_f1 = 0.0

for epoch in range(start_epoch, EPOCHS):
    # Unfreeze backbone at UNFREEZE_EPOCH
    if (epoch == UNFREEZE_EPOCH) and (not backbone_unfrozen):
        print(f"Unfreezing backbone at epoch {epoch+1}...")
        model.unfreeze_backbone()
        backbone_unfrozen = True

        # Just ensure backbone params are trainable; DO NOT recreate optimizer/scheduler
        for p in backbone_params:
            p.requires_grad = True

    train_one_epoch(epoch)
    scheduler.step()

    f1 = evaluate()
    best_f1 = max(best_f1, f1)

    model_to_save_state = model._orig_mod if hasattr(model, '_orig_mod') else model

    torch.save({
        "epoch": epoch,
        "model": model_to_save_state.state_dict(),
        "optimizer": opt.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "backbone_unfrozen": backbone_unfrozen
    }, resume_path)

    print(f"Checkpoint saved: {resume_path}")
    print(f"Best F1 so far: {best_f1:.4f}")


In [ ]:
print("Final evaluation on full test set:")
final_f1 = evaluate()
print("Final F1:", final_f1)
